# 消息处理

## 学习目标
- 理解消息 API 格式
- 掌握并理解模型响应对象
- 构建一个简单的多轮对话聊天机器人

## 基础设置
我们首先导入所需的包并初始化一个客户端对象。
有关如何获取 API 密钥并妥善存储的详细信息，请参阅上一个教程。

In [6]:
from dotenv import load_dotenv
from anthropic import Anthropic

#load environment variable
load_dotenv()

#automatically looks for an "ANTHROPIC_API_KEY" environment variable
client = Anthropic()

## 消息格式

正如我们在上一课中看到的，我们可以使用 `client.messages.create()` 向 Claude 发送消息并获得响应：

In [5]:
response = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "What flavors are used in Dr. Pepper?"}
    ]
)

print(response)

Message(id='msg_013wVsHLHRjuDM2WgvVJ8RNm', content=[ContentBlock(text='The exact flavor formula for Dr Pepper is a closely guarded trade secret, but here are some of the main flavors that are believed to be used:\n\n- Cherry - This is one of the most prominent flavors in Dr Pepper. The cherry flavor comes from the use of a type of cherry extract.\n\n- Prune - Dr Pepper contains a prune-like flavor which contributes to its unique profile.\n\n- Vanilla - Vanilla is another key component that helps round out the flavor.\n\n- Spices - Various spices like cinnamon, prune, and other aromatics are believed to be part of the blend.\n\n- Citrus - Flavors like orange, lemon, and prune add some citrus notes.\n\nThe exact combination of these and other secret ingredients is what gives Dr Pepper its signature taste that differentiates it from other cola or soda flavors. The complex blend of sweet, spicy, and tart notes is part of what makes Dr Pepper a unique and iconic soft drink flavor.', type='t

让我们仔细看看这段代码：
```py
messages=[
        {"role": "user", "content": "What flavors are used in Dr. Pepper?"}
    ]
```

messages 参数是与 Claude API 交互的关键部分。它允许你提供对话历史和上下文，让 Claude 生成相关的响应。

messages 参数期望一个消息字典列表，其中每个字典代表对话中的单条消息。
每个消息字典应包含以下键：

* `role`：一个字符串，表示消息发送者的角色。它可以是 "user"（用户发送的消息）或 "assistant"（Claude 发送的消息）。
* `content`：一个字符串或内容字典列表，表示消息的实际内容。如果提供字符串，它将被视为单个文本内容块。如果提供内容字典列表，每个字典应包含一个 "type"（例如 "text" 或 "image"）和相应的内容。目前，我们暂时将 `content` 保持为单个字符串。

以下是一个包含单条用户消息的 messages 列表示例：

```py
messages = [
    {"role": "user", "content": "Hello Claude! How are you today?"}
]
```

以下是一个包含多条消息的示例，代表一段对话：

```py
messages = [
    {"role": "user", "content": "Hello Claude! How are you today?"},
    {"role": "assistant", "content": "Hello! I'm doing well, thank you. How can I assist you today?"},
    {"role": "user", "content": "Can you tell me a fun fact about ferrets?"},
    {"role": "assistant", "content": "Sure! Did you know that excited ferrets make a clucking vocalization known as 'dooking'?"},
]
```

请记住，消息总是在 user 和 assistant 消息之间交替。



消息格式允许我们以对话形式向 Claude 发出 API 调用，支持**上下文保持**：消息格式允许维护完整的对话历史，包括用户和助手的消息。这确保了 Claude 在生成响应时能够访问对话的完整上下文，从而产生更连贯和相关的输出。

**注意：许多用例不需要对话历史，提供只包含单条消息的消息列表也是完全可以的！**

***

## 小测验

每条消息中两个必需的键是什么？

* **a)** "sender" 和 "text"
* **b)** "role" 和 "content"
* **c)** "user" 和 "assistant"
* **d)** "input" 和 "output"

<details>
  <summary>查看测验答案</summary>
  
  **正确答案是 b。每条消息都应该有 "role" 和 "content"**

</details>




***

## 检查消息响应
接下来，让我们看看从 Claude 返回的响应结构。

让我们让 Claude 做一件简单的事情：

In [7]:
response = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Translate hello to French. Respond with a single word"}
    ]
)

现在让我们检查返回的 `response` 内容：

In [8]:
response

Message(id='msg_01SuDqJSTJaRpkDmHGrbfxCt', content=[ContentBlock(text='Bonjour.', type='text')], model='claude-3-haiku-20240307', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(input_tokens=19, output_tokens=8))

我们得到一个 `Message` 对象，它包含几个属性。以下是一个示例：

```
Message(id='msg_01Mq5gDnUmDESukTgwPV8xtG', content=[TextBlock(text='Bonjour.', type='text')], model='claude-3-haiku-20240307', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(input_tokens=19, output_tokens=8))
```

 最重要的信息是 `content` 属性：它包含模型为我们生成的实际内容。这是一个**内容块列表**，每个内容块都有一个决定其形状的类型。

 

 为了访问模型回复的实际文本内容，我们需要执行以下操作：



In [9]:
print(response.content[0].text)

Bonjour.


除了 `content` 之外，`Message` 对象还包含其他一些信息：

* `id` - 一个唯一的对象标识符
* `type` - 对象类型，始终为 "message"
* `role` - 生成消息的对话角色，始终为 "assistant"
* `model` - 处理请求并生成响应的模型
* `stop_reason` - 模型停止生成的原因。我们稍后会详细了解。
* `stop_sequence` - 我们稍后会详细了解。
* `usage` - 关于计费和速率限制的信息。包含：
    * `input_tokens` - 使用的输入 token 数量。
    * `output_tokens` - 使用的输出 token 数量。

重要的是要知道我们可以访问这些信息，但如果你只记住一件事，请记住：`content` 包含模型生成的实际内容

***
## 练习

编写一个名为 translate 的函数，它接受两个参数：
* 一个单词
* 一种语言

当你调用 `translate` 函数时，它应该返回让 Claude 将 `word` 翻译成 `language` 的结果。例如：

```py
translate("hello", "Spanish")
# 'The word "hello" translated into Spanish is: Hola'

translate("chicken", "Italian")
# 'The Italian word for "chicken" is: pollo'
```

如果你能让 Claude 只回复翻译后的单词而没有任何开场白，可以获得加分：

```py
translate("chicken", "Italian")
# 'pollo'
```


<details>
  <summary>查看练习解答</summary>
  
  以下是一种可能的解答：
  
  ```py
  def translate(word, language):
    response = client.messages.create(
        model="claude-3-opus-20240229",
        max_tokens=1000,
        messages=[
            {"role": "user", "content": f"Translate the word {word} into {language}.  Only respond with the translated word, nothing else"}
        ]
    )
    return response.content[0].text 
  ```

</details>




***

## 消息列表错误

### 错误 #1：以助手消息开始

刚开始使用时，在处理 `messages` 列表时容易犯错误。消息列表必须以 `user` 消息开头。以下代码会产生错误，因为消息列表以助手消息开头：

In [10]:
response = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=1000,
    messages=[
        {"role": "assistant", "content": "Hello there!"}
    ]
)
print(response.content[0].text)

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages: first message must use the "user" role'}}

### 错误 #2：消息交替不正确

消息必须在 `user` 和 `assistant` 之间交替，如果我们不遵循这个规则，就会收到错误：

In [12]:
response = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Hey there!"},
        {"role": "assistant", "content": "Hi there!"},
        {"role": "assistant", "content": "How can I help you??"}
    ]
)
print(response.content[0].text)

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages: roles must alternate between "user" and "assistant", but found multiple "assistant" roles in a row'}}

## 消息列表使用场景



### 让 Claude 替我们说话

另一种获得非常具体输出的常用策略是"让 Claude 替我们说话"。除了向 Claude 提供 `user` 消息外，我们还可以提供一条 `assistant` 消息，Claude 会在生成输出时使用这条消息。

使用 Anthropic 的 API 时，你不仅限于 `user` 消息。如果你提供一条 `assistant` 消息，Claude 会从上一个 `assistant` 标记继续对话。只要记住，我们必须以 `user` 消息开头。

假设我希望 Claude 写一首以"calming mountain air"（宁静的山风）开头的俳句。我可以提供以下对话历史：

```py
messages=[
        {"role": "user", "content": f"Generate a beautiful haiku"},
        {"role": "assistant", "content": "calming mountain air"}
    ]
```
我们告诉 Claude 我们希望它生成一首俳句，并且把俳句的第一行放到 Claude 嘴里




In [10]:
response = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=500,
    messages=[
        {"role": "user", "content": f"Generate a beautiful haiku"},
        {"role": "assistant", "content": "calming mountain air"}
    ]
)
print(response.content[0].text)

,
dancing sunlight on still waters,
nature's gentle grace.


要获得完整的俳句，从我们提供的行开始：

In [11]:
print("calming mountain air" + response.content[0].text)

calming mountain air,
dancing sunlight on still waters,
nature's gentle grace.


### 少样本提示

最有用的提示策略之一被称为"少样本提示"，它涉及向模型提供少量**示例**。这些示例有助于引导 Claude 的生成输出。消息对话历史是向 Claude 提供示例的简单方式。

例如，假设我们想用 Claude 来分析推文中的情绪。我们可以简单地向 Claude 请求"请分析这条推文的情绪："，然后观察我们得到什么样的输出：

In [18]:
response = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=500,
    messages=[
        {"role": "user", "content": f"Analyze the sentiment in this tweet: Just tried the new spicy pickles from @PickleCo, and my taste buds are doing a happy dance! 🌶️🥒 #pickleslove #spicyfood"},
    ]
)
print(response.content[0].text)

第一次运行上面的代码时，Claude 生成了这样的长回复：
```
The sentiment in this tweet is overwhelmingly positive. The user expresses their enjoyment of the new spicy pickles from @PickleCo, using enthusiastic language and emojis to convey their delight.

Positive indicators:
1. "My taste buds are doing a happy dance!" - This phrase indicates that the user is extremely pleased with the taste of the pickles, to the point of eliciting a joyful physical response.

2. Emojis - The use of the hot pepper 🌶️ and cucumber 🥒 emojis further emphasizes the user's excitement about the spicy pickles.

3. Hashtags - The inclusion of #pickleslove and #spicyfood hashtags suggests that the user has a strong affinity for pickles and spicy food, and the new product aligns perfectly with their preferences.

4. Exclamation mark - The exclamation mark at the end of the first sentence adds emphasis to the user's positive experience.

Overall, the tweet conveys a strong sense of satisfaction, excitement, and enjoyment related to trying the new spicy pickles from @PickleCo.
```

这是一个很好的回复，但它可能包含了比我们需要的更多的信息，特别是当我们试图自动化大量推文的情绪分析时。

我们可能更喜欢 Claude 以标准化的输出格式回复，比如一个词（POSITIVE、NEUTRAL、NEGATIVE）或一个数值（1、0、-1）。为了可读性和简单性，让我们让 Claude 回答"POSITIVE"或"NEGATIVE"。实现这一点的方法之一是通过少样本提示。我们可以向 Claude 提供一个对话历史，展示我们希望它如何回复：

```py
messages=[
        {"role": "user", "content": "Unpopular opinion: Pickles are disgusting. Don't @ me"},
        {"role": "assistant", "content": "NEGATIVE"},
        {"role": "user", "content": "I think my love for pickles might be getting out of hand. I just bought a pickle-shaped pool float"},
        {"role": "assistant", "content": "POSITIVE"},
        {"role": "user", "content": "Seriously why would anyone ever eat a pickle?  Those things are nasty!"},
        {"role": "assistant", "content": "NEGATIVE"},
        {"role": "user", "content": "Just tried the new spicy pickles from @PickleCo, and my taste buds are doing a happy dance! 🌶️🥒 #pickleslove #spicyfood"},
    ]
```



In [21]:
response = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=500,
    messages=[
        {"role": "user", "content": "Unpopular opinion: Pickles are disgusting. Don't @ me"},
        {"role": "assistant", "content": "NEGATIVE"},
        {"role": "user", "content": "I think my love for pickles might be getting out of hand. I just bought a pickle-shaped pool float"},
        {"role": "assistant", "content": "POSITIVE"},
        {"role": "user", "content": "Seriously why would anyone ever eat a pickle?  Those things are nasty!"},
        {"role": "assistant", "content": "NEGATIVE"},
        {"role": "user", "content": "Just tried the new spicy pickles from @PickleCo, and my taste buds are doing a happy dance! 🌶️🥒 #pickleslove #spicyfood"},
    ]
)
print(response.content[0].text)

POSITIVE


***

## 练习

### 你的任务：构建一个聊天机器人

构建一个简单的多轮命令行聊天机器人脚本。消息格式非常适合构建基于聊天的应用程序。用 Claude 构建聊天机器人很简单：

1. 保留一个列表来存储对话历史
2. 使用 `input()` 询问用户输入，并将用户输入添加到消息列表
3. 将消息历史发送给 Claude
4. 将 Claude 的响应打印给用户
5. 将 Claude 的助手响应添加到历史记录中
6. 返回步骤 2 重复！（使用循环，并提供用户退出的方式）


<details>
    <summary>查看练习解答</summary>

    ```py

    conversation_history = []

    while True:
        user_input = input("User: ")
        
        if user_input.lower() == "quit":
            print("Conversation ended.")
            break
        
        conversation_history.append({"role": "user", "content": user_input})
    
        response = client.messages.create(
            model="claude-3-haiku-20240307",
            messages=conversation_history,
            max_tokens=500
        )
    
        assistant_response = response.content[0].text
        print(f"Assistant: {assistant_response}")
        conversation_history.append({"role": "assistant", "content": assistant_response})
    ```
</details>

***
